# **Data Preprocessing**

This notebook prepares the EEG dataset for modeling by applying necessary preprocessing steps. It ensures that the data is clean, consistently structured, and in a format suitable for machine learning, including handling feature types, encoding categorical variables, and assembling the final feature set.

### **Import Libraries**

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler
)

### **Initialize Spark Session**

In [2]:
spark = (
    SparkSession.builder
    .appName("EEG_Data_Preprocessing")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 21:57:51 WARN Utils: Your hostname, Sarras-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.134 instead (on interface en0)
26/04/13 21:57:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 21:57:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **Define Paths**

In [3]:
project_root = Path("..").resolve()

input_dir = project_root / "data" / "processed" / "feature_engineered" / "final_modeling_table"
demographic_path = project_root / "data" / "demographic.csv"

output_dir = project_root / "data" / "processed" / "preprocessed_final_modeling_table"
train_output_dir = output_dir / "train"
test_output_dir = output_dir / "test"
pipeline_dir = project_root / "data" / "processed" / "preprocessing_pipeline"

output_dir.mkdir(parents=True, exist_ok=True)
pipeline_dir.mkdir(parents=True, exist_ok=True)

### **Load the Engineered Modeling Table**

In [4]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(input_dir))
)

print(f"Number of columns in modeling table: {len(df.columns)}")
df.select("subject", "trial", "condition").show(5, truncate=False)

Number of columns in modeling table: 174
+-------+-----+---------+
|subject|trial|condition|
+-------+-----+---------+
|49.0   |1.0  |1.0      |
|49.0   |1.0  |2.0      |
|49.0   |1.0  |3.0      |
|49.0   |2.0  |1.0      |
|49.0   |2.0  |2.0      |
+-------+-----+---------+
only showing top 5 rows


### **Load the Demographic Table**

In [5]:
demographic_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(demographic_path))
)

demographic_df.show(5, truncate=False)

+-------+------+-------+----+----------+
|subject| group| gender| age| education|
+-------+------+-------+----+----------+
|1      |0.0   | M     |44.0|16.0      |
|2      |0.0   | M     |39.0|17.0      |
|3      |0.0   | M     |53.0|18.0      |
|4      |0.0   | M     |52.0|15.0      |
|5      |0.0   | M     |41.0|16.0      |
+-------+------+-------+----+----------+
only showing top 5 rows


### **Remove Any Duplicate Rows**

In [6]:
df = df.dropDuplicates(["subject", "trial", "condition"])

### **Clean and Prepare Demographics Columns**

In [7]:
print(demographic_df.columns)

['subject', ' group', ' gender', ' age', ' education']


In [8]:
demographic_df = demographic_df.dropDuplicates(["subject"])

demographic_df = (
    demographic_df
    .withColumnRenamed(" group", "label")
    .withColumnRenamed(" gender", "gender")
    .withColumnRenamed(" age", "age")
    .withColumnRenamed(" education", "education")
)

demographic_df = demographic_df.select(
    "subject",
    "label",
    "gender",
    "age",
    "education",
)

### **Join the Target Labels to the Trial-Level Table**

In [9]:
df = df.join(
    demographic_df,
    on="subject",
    how="left"
)

### **Validate the Target Join**

In [10]:
df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "gender",
    "age",
    "education",
).show(10, truncate=False)


+-------+-----+---------+-----+------+----+---------+
|subject|trial|condition|label|gender|age |education|
+-------+-----+---------+-----+------+----+---------+
|49.0   |19.0 |1.0      |1.0  | M    |22.0|11.0     |
|47.0   |6.0  |3.0      |1.0  | M    |41.0|16.0     |
|47.0   |10.0 |3.0      |1.0  | M    |41.0|16.0     |
|42.0   |19.0 |1.0      |1.0  | M    |45.0|14.0     |
|44.0   |52.0 |1.0      |1.0  | M    |47.0|13.0     |
|44.0   |70.0 |1.0      |1.0  | M    |47.0|13.0     |
|45.0   |24.0 |2.0      |1.0  | M    |51.0|13.0     |
|45.0   |65.0 |2.0      |1.0  | M    |51.0|13.0     |
|46.0   |6.0  |1.0      |1.0  | M    |40.0|14.0     |
|46.0   |31.0 |1.0      |1.0  | M    |40.0|14.0     |
+-------+-----+---------+-----+------+----+---------+
only showing top 10 rows


### **Identify Missing Values per Column**

In [11]:
missing_summary = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_summary.show(truncate=False)

26/04/13 21:58:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----+---------+---------+---------+-----------+-----------+----------+----------+------------+------------+---------+---------+-----------+-----------+----------+----------+------------+------------+----------+----------+------------+------------+---------+---------+-----------+-----------+---------+---------+-----------+-----------+----------+----------+------------+------------+----------+----------+------------+------------+--------------+--------------+--------------------+---------------+-----------+------------+-------------+--------------+--------------+--------------------+---------------+-----------+------------+-------------+----------------+----------------+----------------------+-----------------+-------------+--------------+---------------+----------------+----------------+----------------------+-----------------+-------------+--------------+---------------+------------------+------------------+-------------------+-------------------+------------------+----------

### **Define Identifier, Target, Categorical, and Numeric Columns**

In [12]:
id_cols = ["subject", "trial"]
target_col = "label"
demographic_categorical_cols = ["gender"]
demographic_numeric_cols = ["age", "education"]
categorical_cols = ["condition"] + demographic_categorical_cols

excluded_cols = set(id_cols + [target_col] + categorical_cols)

numeric_cols = [
    field.name
    for field in df.schema.fields
    if field.name not in excluded_cols
    and isinstance(
        field.dataType,
        (
            T.IntegerType,
            T.LongType,
            T.FloatType,
            T.DoubleType,
            T.ShortType,
            T.DecimalType
        )
    )
]

missing_demographic_numeric_cols = [
    c for c in demographic_numeric_cols
    if c not in numeric_cols
]
if missing_demographic_numeric_cols:
    raise ValueError(
        f"Demographic numeric columns missing from numeric feature set: {missing_demographic_numeric_cols}"
    )

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Demographic categorical columns encoded into features: {demographic_categorical_cols}")
print(f"Demographic numeric columns scaled inside features: {demographic_numeric_cols}")

Categorical columns: ['condition', 'gender']
Numeric columns: 173
Demographic categorical columns encoded into features: ['gender']
Demographic numeric columns scaled inside features: ['age', 'education']


### **Convert Categorical Columns to String Type**

In [13]:
for c in categorical_cols:
    df = df.withColumn(c, F.col(c).cast("string"))

subject_df = df.select("subject").distinct()
train_subjects_df, test_subjects_df = subject_df.randomSplit([0.8, 0.2], seed=42)

train_df = df.join(train_subjects_df, on="subject", how="inner")
test_df = df.join(test_subjects_df, on="subject", how="inner")

print(f"Train subjects: {train_subjects_df.count()}")
print(f"Test subjects: {test_subjects_df.count()}")
print(f"Train rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

Train subjects: 64
Test subjects: 17
Train rows: 18305
Test rows: 4896


### **Define the Categorical Encoding Stage**

In [14]:
indexer_stages = [
    StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_indexed",
        handleInvalid="keep"
    )
    for col_name in categorical_cols
]

### **Define the One-Hot Encoding Stage**

In [15]:
encoded_input_cols = [f"{c}_indexed" for c in categorical_cols]
encoded_output_cols = [f"{c}_ohe" for c in categorical_cols]

encoder_stage = OneHotEncoder(
    inputCols=encoded_input_cols,
    outputCols=encoded_output_cols,
    handleInvalid="keep"
)

### **Define the Feature Assembly Stage**

In [16]:
feature_input_cols = numeric_cols + encoded_output_cols

print(f"Encoded categorical feature columns: {encoded_output_cols}")
print(
    "Demographic feature inputs included in features_raw:",
    [c for c in demographic_numeric_cols if c in feature_input_cols] + encoded_output_cols
)

assembler_stage = VectorAssembler(
    inputCols=feature_input_cols,
    outputCol="features_raw",
    handleInvalid="error"
)

Encoded categorical feature columns: ['condition_ohe', 'gender_ohe']
Demographic feature inputs included in features_raw: ['age', 'education', 'condition_ohe', 'gender_ohe']


### **Define the Feature Scaling Stage**

In [17]:
scaler_stage = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

### **Build the Complete Preprocessing Pipeline**

In [18]:
preprocessing_pipeline = Pipeline(
    stages=indexer_stages + [
        encoder_stage,
        assembler_stage,
        scaler_stage
    ]
)

### **Fit the Preprocessing Pipeline and Transform the Dataset**

In [19]:
preprocessing_model = preprocessing_pipeline.fit(train_df)
preprocessed_train_df = preprocessing_model.transform(train_df)
preprocessed_test_df = preprocessing_model.transform(test_df)

### **Review the Preprocessed Output**

In [20]:
print("Train preview")
preprocessed_train_df.select(
    "subject",
    "trial",
    "condition",
    "gender",
    "label",
    "features_raw",
    "features"
).show(5, truncate=False)

print("Test preview")
preprocessed_test_df.select(
    "subject",
    "trial",
    "condition",
    "gender",
    "label",
    "features_raw",
    "features"
).show(5, truncate=False)

Train preview


+-------+-----+---------+------+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+-------+-----+---------+------+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### **Build the Final Preprocessed Modeling Table**

In [21]:
final_preprocessed_train_df = preprocessed_train_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "gender",
    "age",
    "education",
    "features"
)

final_preprocessed_test_df = preprocessed_test_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "gender",
    "age",
    "education",
    "features"
)

In [22]:
print("Train output preview")
final_preprocessed_train_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "gender",
    "age",
    "education",
).show(10, truncate=False)

print("Test output preview")
final_preprocessed_test_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "gender",
    "age",
    "education",
).show(10, truncate=False)

Train output preview
+-------+-----+---------+-----+------+----+---------+
|subject|trial|condition|label|gender|age |education|
+-------+-----+---------+-----+------+----+---------+
|49.0   |19.0 |1.0      |1.0  | M    |22.0|11.0     |
|42.0   |19.0 |1.0      |1.0  | M    |45.0|14.0     |
|44.0   |52.0 |1.0      |1.0  | M    |47.0|13.0     |
|44.0   |70.0 |1.0      |1.0  | M    |47.0|13.0     |
|45.0   |24.0 |2.0      |1.0  | M    |51.0|13.0     |
|45.0   |65.0 |2.0      |1.0  | M    |51.0|13.0     |
|43.0   |72.0 |2.0      |1.0  | M    |49.0|13.0     |
|43.0   |78.0 |2.0      |1.0  | M    |49.0|13.0     |
|43.0   |91.0 |2.0      |1.0  | M    |49.0|13.0     |
|49.0   |65.0 |1.0      |1.0  | M    |22.0|11.0     |
+-------+-----+---------+-----+------+----+---------+
only showing top 10 rows
Test output preview
+-------+-----+---------+-----+------+----+---------+
|subject|trial|condition|label|gender|age |education|
+-------+-----+---------+-----+------+----+---------+
|47.0   |6.0  |3

### **Save the Final Preprocessed Dataset**

In [23]:
(
    final_preprocessed_train_df
    .write
    .mode("overwrite")
    .parquet(str(train_output_dir))
)

(
    final_preprocessed_test_df
    .write
    .mode("overwrite")
    .parquet(str(test_output_dir))
)

26/04/13 21:59:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/13 21:59:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/13 21:59:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/13 21:59:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/13 21:59:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/13 21:59:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/13 21:59:05 WARN MemoryManager: Total allocation exceeds 95.00%

### **Save the Fitted Preprocessing Pipeline**

In [24]:
preprocessing_model.write().overwrite().save(str(pipeline_dir))

### **Stop Spark Session**

In [27]:
spark.stop()

## **Spark PCA on the Combined Reduced Raw + Engineered Feature Set**

This section adds Spark-based PCA to the combined reduced raw EEG table and the engineered feature table. PCA is dimensionality reduction; it creates new principal components from the full joined reduced raw + engineered feature set instead of selecting a subset of the original columns.

In [25]:
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path

from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql import SparkSession
from pyspark.sql import types as T

from pyspark.ml.feature import PCA, StandardScaler, VectorAssembler
from pyspark.ml.linalg import VectorUDT


def get_or_create_pca_spark():
    existing_spark = globals().get("spark")
    if existing_spark is not None:
        try:
            existing_spark.range(1).count()
            print("Reusing the active SparkSession for the PCA section.")
            return existing_spark, True
        except Exception:
            pass

    pca_spark = (
        SparkSession.builder
        .appName("EEG_Data_Preprocessing_PCA")
        .getOrCreate()
    )
    print("Started a SparkSession for the PCA section.")
    return pca_spark, False


spark, reused_active_spark = get_or_create_pca_spark()

project_root = Path(globals().get("project_root", Path("..").resolve()))
engineered_input_dir = Path(
    globals().get(
        "input_dir",
        project_root / "data" / "processed" / "feature_engineered" / "final_modeling_table"
    )
)
preferred_reduced_raw_input_dir = project_root / "data" / "all_csv_agg_ALLLL"
fallback_reduced_raw_input_dir = project_root / "data" / "processed" / "all_csv_agg_ALLLL"
reduced_raw_input_dir = (
    preferred_reduced_raw_input_dir
    if preferred_reduced_raw_input_dir.exists()
    else fallback_reduced_raw_input_dir
)
demographic_path = Path(
    globals().get(
        "demographic_path",
        project_root / "data" / "demographic.csv"
    )
)
pca_output_dir = project_root / "data" / "processed" / "pca_final_modeling_table"

print(f"Engineered input: {engineered_input_dir}")
print(f"Reduced raw input: {reduced_raw_input_dir}")
print(f"PCA output: {pca_output_dir}")

Reusing the active SparkSession for the PCA section.
Engineered input: /Users/sarrachouk/Desktop/EEG-Schizophrenia/data/processed/feature_engineered/final_modeling_table
Reduced raw input: /Users/sarrachouk/Desktop/EEG-Schizophrenia/data/processed/all_csv_agg_ALLLL
PCA output: /Users/sarrachouk/Desktop/EEG-Schizophrenia/data/processed/pca_final_modeling_table


### **Load the Reduced Raw and Engineered Tables**

The PCA section reuses the existing engineered trial-level table when it is still valid in memory.

In [26]:
preferred_join_keys = ["subject", "trial", "condition"]


def trim_column_names_if_needed(spark_df, label):
    rename_pairs = [(c, c.strip()) for c in spark_df.columns if c != c.strip()]
    trimmed_df = spark_df
    for old_name, new_name in rename_pairs:
        trimmed_df = trimmed_df.withColumnRenamed(old_name, new_name)
    if rename_pairs:
        print(f"Trimmed whitespace from {len(rename_pairs)} {label} columns.")
    else:
        print(f"{label} columns were already clean.")
    return trimmed_df


def reuse_dataframe(var_name, required_cols=None):
    candidate = globals().get(var_name)
    if not isinstance(candidate, SparkDataFrame):
        return None
    if required_cols and not set(required_cols).issubset(candidate.columns):
        return None
    try:
        candidate.limit(1).count()
        return candidate
    except Exception:
        return None


engineered_df = (
    reuse_dataframe("df", required_cols=preferred_join_keys)
    if reused_active_spark else None
)
if engineered_df is not None:
    print("Reusing the existing engineered base table from `df`.")
else:
    engineered_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(engineered_input_dir))
    )
    print("Loaded the engineered table from disk.")

engineered_df = trim_column_names_if_needed(engineered_df, "Engineered table")

engineered_non_feature_cols = [
    c for c in ["label", "gender", "age", "education"]
    if c in engineered_df.columns
]
engineered_feature_df = (
    engineered_df.drop(*engineered_non_feature_cols)
    if engineered_non_feature_cols else engineered_df
)

reduced_raw_df = (
    reuse_dataframe("reduced_raw_df", required_cols=preferred_join_keys)
    if reused_active_spark else None
)
if reduced_raw_df is not None:
    print("Reusing the existing reduced raw table from `reduced_raw_df`.")
else:
    reduced_raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(reduced_raw_input_dir))
    )
    print("Loaded the reduced raw table from disk.")

reduced_raw_df = trim_column_names_if_needed(reduced_raw_df, "Reduced raw table")

print(f"Engineered feature columns available for PCA join: {len(engineered_feature_df.columns)}")
print(f"Reduced raw columns available for PCA join: {len(reduced_raw_df.columns)}")

Reusing the existing engineered base table from `df`.
Engineered table columns were already clean.


Loaded the reduced raw table from disk.
Reduced raw table columns were already clean.
Engineered feature columns available for PCA join: 174
Reduced raw columns available for PCA join: 1543


### **Join the Two Tables and Keep a Single Target Column**

The join stays at the row level by using `subject`, `trial`, and `condition`. Overlapping non-key feature columns from the reduced raw table are dropped before the join so the combined table keeps one copy of each feature, and `label` is only reattached if it is not already available.

In [27]:
join_keys = [
    c for c in preferred_join_keys
    if c in engineered_feature_df.columns and c in reduced_raw_df.columns
]
if join_keys != preferred_join_keys:
    raise ValueError(f"Expected join keys {preferred_join_keys}, but found {join_keys}.")

duplicate_raw_feature_cols = [
    c for c in reduced_raw_df.columns
    if c not in join_keys and c in engineered_feature_df.columns
]

reduced_raw_for_join_df = reduced_raw_df.select(
    *join_keys,
    *[
        c for c in reduced_raw_df.columns
        if c not in join_keys + duplicate_raw_feature_cols
    ]
)

combined_df = engineered_feature_df.join(
    reduced_raw_for_join_df,
    on=join_keys,
    how="inner"
)

if "label" in engineered_df.columns:
    label_source_df = engineered_df.select(*join_keys, "label").dropDuplicates(join_keys)
    combined_df = combined_df.join(label_source_df, on=join_keys, how="left")
    print("Reused the existing `label` column from the engineered-side DataFrame.")
else:
    demographic_source_df = (
        reuse_dataframe("demographic_df", required_cols=["subject"])
        if reused_active_spark else None
    )
    if demographic_source_df is None or "label" not in demographic_source_df.columns:
        demographic_source_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(str(demographic_path))
        )
        demographic_source_df = trim_column_names_if_needed(
            demographic_source_df,
            "Demographic table"
        )
        if "group" in demographic_source_df.columns and "label" not in demographic_source_df.columns:
            demographic_source_df = demographic_source_df.withColumnRenamed("group", "label")
    label_source_df = demographic_source_df.select("subject", "label").dropDuplicates(["subject"])
    combined_df = combined_df.join(label_source_df, on="subject", how="left")
    print("Joined `label` from the demographic table because it was not already attached.")

remaining_cols = [
    c for c in combined_df.columns
    if c not in join_keys + ["label"]
]
combined_df = combined_df.select(*join_keys, "label", *remaining_cols)

print(f"Join keys: {join_keys}")
print(f"Dropped duplicate raw feature columns before join: {len(duplicate_raw_feature_cols)}")
print(f"Combined columns after join: {len(combined_df.columns)}")

Reused the existing `label` column from the engineered-side DataFrame.
Join keys: ['subject', 'trial', 'condition']
Dropped duplicate raw feature columns before join: 36
Combined columns after join: 1679


### **Define PCA Inputs, Assemble the Feature Vector, and Scale It**

Only numeric feature columns from the joined table are sent into PCA. Identifier fields, the target column, and any existing vector columns are excluded so PCA is fit on a clean Spark ML feature vector built from the combined feature table.

In [28]:
non_feature_cols = set(join_keys + ["label"])
vector_cols = [
    field.name
    for field in combined_df.schema.fields
    if isinstance(field.dataType, VectorUDT)
]

numeric_pca_input_cols = [
    field.name
    for field in combined_df.schema.fields
    if field.name not in non_feature_cols
    and field.name not in vector_cols
    and isinstance(
        field.dataType,
        (
            T.IntegerType,
            T.LongType,
            T.FloatType,
            T.DoubleType,
            T.ShortType,
            T.DecimalType
        )
    )
]

if not numeric_pca_input_cols:
    raise ValueError("No numeric feature columns were available for PCA.")

pca_base_df = combined_df.select(*join_keys, "label", *numeric_pca_input_cols)

assembler = VectorAssembler(
    inputCols=numeric_pca_input_cols,
    outputCol="combinedFeaturesRaw",
    handleInvalid="error"
)

scaler = StandardScaler(
    inputCol="combinedFeaturesRaw",
    outputCol="combinedFeaturesScaled",
    withMean=True,
    withStd=True
)

assembled_df = assembler.transform(pca_base_df)
scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df).select(
    *join_keys,
    "label",
    "combinedFeaturesScaled"
)

print(f"Numeric PCA input columns: {len(numeric_pca_input_cols)}")
print(f"Existing vector columns excluded from PCA inputs: {vector_cols}")

26/04/13 21:59:31 WARN DAGScheduler: Broadcasting large task binary with size 1188.0 KiB


Numeric PCA input columns: 1535
Existing vector columns excluded from PCA inputs: []


26/04/13 21:59:34 WARN DAGScheduler: Broadcasting large task binary with size 1265.9 KiB


### **Fit PCA and Inspect Explained Variance**

This PCA fit evaluates the full combined feature space to identify the smallest number of principal components needed to reach the target cumulative explained variance.

In [29]:
inspection_k = len(numeric_pca_input_cols)
variance_target = 0.95

inspection_pca = PCA(
    k=inspection_k,
    inputCol="combinedFeaturesScaled",
    outputCol="pcaInspectionFeatures"
)
inspection_pca_model = inspection_pca.fit(scaled_df)

explained_variance_pdf = pd.DataFrame({
    "component": range(1, inspection_k + 1),
    "explained_variance_ratio": inspection_pca_model.explainedVariance.toArray()
})
explained_variance_pdf["cumulative_explained_variance"] = (
    explained_variance_pdf["explained_variance_ratio"].cumsum()
)

selected_k_candidates = explained_variance_pdf.loc[
    explained_variance_pdf["cumulative_explained_variance"] >= variance_target,
    "component"
]

if selected_k_candidates.empty:
    raise ValueError(
        f"Unable to reach {variance_target:.0%} cumulative explained variance with {inspection_k} components."
    )

selected_k = int(selected_k_candidates.iloc[0])
selected_variance = float(
    explained_variance_pdf.loc[
        explained_variance_pdf["component"] == selected_k,
        "cumulative_explained_variance"
    ].iloc[0]
)

if selected_k == inspection_k:
    pca_transformed_df = (
        inspection_pca_model
        .transform(scaled_df)
        .withColumnRenamed("pcaInspectionFeatures", "pcaFeatures")
    )
else:
    final_pca = PCA(
        k=selected_k,
        inputCol="combinedFeaturesScaled",
        outputCol="pcaFeatures"
    )
    final_pca_model = final_pca.fit(scaled_df)
    pca_transformed_df = final_pca_model.transform(scaled_df)

print(f"Evaluated all {inspection_k} principal components in the combined feature space.")
print(
    f"Selected {selected_k} principal components to retain {selected_variance:.2%} cumulative explained variance."
)

explained_variance_pdf

26/04/13 21:59:40 WARN DAGScheduler: Broadcasting large task binary with size 1226.6 KiB
26/04/13 21:59:41 WARN DAGScheduler: Broadcasting large task binary with size 1226.6 KiB
26/04/13 21:59:41 WARN DAGScheduler: Broadcasting large task binary with size 1230.6 KiB
26/04/13 21:59:43 WARN DAGScheduler: Broadcasting large task binary with size 1231.7 KiB
26/04/13 21:59:43 WARN DAGScheduler: Broadcasting large task binary with size 1227.1 KiB
26/04/13 21:59:44 WARN DAGScheduler: Broadcasting large task binary with size 1228.8 KiB
26/04/13 21:59:45 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/13 21:59:52 WARN DAGScheduler: Broadcasting large task binary with size 1229.7 KiB
26/04/13 21:59:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/04/13 22:00:04 WARN DAGScheduler: Broadcasting large task binary with size 1226.6 KiB
26/04/13 22:00:04 WARN DAGScheduler: Broadcasting large task binary wi

Evaluated all 1535 principal components in the combined feature space.
Selected 259 principal components to retain 95.02% cumulative explained variance.


,component,explained_variance_ratio,cumulative_explained_variance
0,1,1.584593e-01,0.158459
1,2,1.364409e-01,0.294900
2,3,6.483323e-02,0.359733
3,4,5.135879e-02,0.411092
4,5,4.213879e-02,0.453231
...,...,...,...
1530,1531,1.421301e-17,1.000000
1531,1532,1.421301e-17,1.000000
1532,1533,1.421301e-17,1.000000
1533,1534,1.044801e-17,1.000000


### **Build the Final PCA Dataset and Save It as a New Output**

The saved PCA dataset keeps the row-level identifiers, the target label, and the Spark vector of principal components.

In [31]:
pca_modeling_df = pca_transformed_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "pcaFeatures"
)

(
    pca_modeling_df
    .write
    .mode("overwrite")
    .parquet(str(pca_output_dir))
)

print(f"Saved the PCA modeling dataset to {pca_output_dir}")

26/04/13 22:00:51 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB
26/04/13 22:00:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/13 22:00:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/13 22:00:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/13 22:00:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/13 22:00:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Saved the PCA modeling dataset to /Users/sarrachouk/Desktop/EEG-Schizophrenia/data/processed/pca_final_modeling_table
